In [1]:
from qiskit import *
from qiskit_aer import Aer
from PIL import Image
import numpy as np
import random

In [ ]:
# Parameters
height, width = 1024, 1024
num_qubits = 3
theta = 0.5
max_val = 2**num_qubits - 1
shots = height * width

qc = QuantumCircuit(num_qubits, num_qubits)

for i in range(num_qubits):
    qc.ry(theta, i)
    qc.barrier()
    if i < num_qubits - 1:
        qc.cx(i, i+1)

qc.measure(range(num_qubits), range(num_qubits))

simulator = Aer.get_backend('qasm_simulator')
qc_transpiled = transpile(qc, simulator)
result = simulator.run(qc_transpiled, backend=simulator, shots=shots).result()
counts = result.get_counts()

bitstrings = []
for b, c in counts.items():
    bitstrings.extend([b] * c)

random.shuffle(bitstrings) # Optional

# Create Image
image = np.zeros((height, width), dtype=np.uint8)
for i in range(height):
    for j in range(width):
        idx = i * width + j
        b = bitstrings[idx]
        intensity = int(b, 2) / max_val * 255
        image[i, j] = int(intensity)

# Show Image
img = Image.fromarray(image, mode='L')
img.show()
img.save("quantum_art_01.png")

qc.draw()

┌─────────┐ ░                  ░                  ░ ┌─┐      
q_0: ┤ Ry(0.5) ├─░───■──────────────░──────────────────░─┤M├──────
     └─────────┘ ░ ┌─┴─┐┌─────────┐ ░                  ░ └╥┘┌─┐   
q_1: ────────────░─┤ X ├┤ Ry(0.5) ├─░───■──────────────░──╫─┤M├───
                 ░ └───┘└─────────┘ ░ ┌─┴─┐┌─────────┐ ░  ║ └╥┘┌─┐
q_2: ────────────░──────────────────░─┤ X ├┤ Ry(0.5) ├─░──╫──╫─┤M├
                 ░                  ░ └───┘└─────────┘ ░  ║  ║ └╥┘
c: 3/═════════════════════════════════════════════════════╩══╩══╩═
                                                          0  1  2

In [3]:
# Parameters
height, width = 15, 15
num_qubits = 2
max_val = 2**num_qubits - 1

simulator = Aer.get_backend('qasm_simulator')

# Initialize Image
image = np.zeros((height, width), dtype=np.uint8)

# Generate 1 circuit per pixe
for i in range(height):
    for j in range(width):
        qc = QuantumCircuit(num_qubits, num_qubits)

        # Use coordinates to ser angles (normalized between 0 and 2pi)
        theta_x = (i / height) * 2 * np.pi
        theta_y = (j / width) * 2 * np.pi

        # Apply Ry gates with different position-based rotations
        qc.ry(theta_x, 0)
        qc.ry(theta_y, 1)

        # Optional: entangle for more complex beahviour
        qc.cx(0, 1)

        qc.measure([0,1], [0,1])

        qc_transpiled = transpile(qc, simulator)
        result = simulator.run(qc_transpiled, backend=simulator, shots=1).result()
        bitstring = list(result.get_counts().keys())[0]

        # Convert         
        value = int(bitstring, 2) / max_val * 255
        image[i, j] = int(value)

# Display the image
img = Image.fromarray(image, mode='L')
img_resized = img.resize((512, 512), resample=Image.NEAREST)
img_resized.show()

In [ ]:
height, width = 15, 15
num_qubits = 2
max_val = 2** num_qubits - 1
simulator = Aer.get_backend('qasm_simulator')

def generate_channel(height, width, theta_shift_x=0, theta_shift_y=0):
    image = np.zeros((height, width), dtype=np.uint8)

    for i in range(height):
        for j in range(width):
            qc = QuantumCircuit(num_qubits, num_qubits)

            theta_x = ((i / height) * 2 * np.pi) + theta_shift_x
            theta_y = ((j / width) * 2 * np.pi) + theta_shift_y

            qc.ry(theta_x, 0)
            qc.ry(theta_y, 1)

            qc.cx(0, 1)

            qc.measure([0, 1], [0, 1])

            qc_transpiled = transpile(qc, simulator)
            result = simulator.run(qc_transpiled, backend=simulator, shots=1).result()
            bitstring = list(result.get_counts().keys())[0]

            # Convert         
            value = int(bitstring, 2) / max_val * 255
            image[i, j] = int(value)

    return image

# Generate R, G, B channels with small variations
r_channel = generate_channel(height, width, theta_shift_x=0.0, theta_shift_y=0.0)
r_channel = generate_channel(height, width, theta_shift_x=0.5, theta_shift_y=0.5)
r_channel = generate_channel(height, width, theta_shift_x=1.0, theta_shift_y=1.0)